![tracker](https://us-central1-vertex-ai-mlops-369716.cloudfunctions.net/pixel-tracking?path=statmike%2Fvertex-ai-mlops%2FApplied+ML%2FAI+Agents%2Fagent-building%2Fdeploy&file=interact.ipynb)
<!--- header table --->
<table>
<tr>
  <td style="text-align: center">
    <a href="https://github.com/statmike/vertex-ai-mlops/blob/main/Applied%20ML/AI%20Agents/agent-building/deploy/interact.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo">
      <br>View on<br>GitHub
    </a>
  </td>
</tr>
<tr>
  <td style="text-align: right">
    <b>Connect With Author On: </b>
    <a href="https://www.linkedin.com/in/statmike"><img src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="Linkedin Logo" width="20px"></a>
    <a href="https://www.github.com/statmike"><img src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub Logo" width="20px"></a>
  </td>
</tr>
</table><br/><br/>

---
# Interact with a Deployed Agent — Sessions & Memory

Phase 2 walkthrough. Once you've deployed the concierge to **Agent Runtime**
(`make deploy-concierge`), this notebook connects to it and demonstrates the
**Scale** pillar features that only exist for a *deployed* agent:

- **Managed sessions** — persistent, cloud-based conversation state.
- **Memory Bank** — long-term facts that survive across sessions.

> Prerequisite: a live deployment. Check with
> `uv run python deploy/deploy.py concierge --info`.

## 1. Connect to the deployment

We read the saved `resource_name` from `deploy/concierge/deployment.json` and
fetch the remote agent. No agent code is imported here — we talk to it purely
over the Agent Runtime API, exactly as another service would.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import dotenv

warnings.filterwarnings("ignore", message=".*EXPERIMENTAL.*")

# Resolve the project root whether this runs from deploy/ or the repo root.
ROOT = Path.cwd()
if (ROOT / "deploy").exists() is False and (ROOT.parent / "deploy").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
dotenv.load_dotenv(ROOT / ".env")

meta = json.loads((ROOT / "deploy" / "concierge" / "deployment.json").read_text())
resource_name = meta["resource_name"]
print("Deployment:", resource_name)

In [ ]:
import os

import vertexai

project = os.environ["GOOGLE_CLOUD_PROJECT"]
location = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
client = vertexai.Client(project=project, location=location)
agent = client.agent_engines.get(name=resource_name)
agent

## 2. A session remembers the current conversation

Create a session, then ask two turns. The second turn relies on state from the
first — the managed session carries it, no client-side history needed.

In [ ]:
import asyncio

USER_ID = "demo_shopper"

async def ask(session_id, message):
    print(f"\nYOU: {message}")
    print("AGENT: ", end="")
    async for event in agent.async_stream_query(
        user_id=USER_ID, session_id=session_id, message=message
    ):
        content = event.get("content")
        if content:
            for part in content.get("parts", []):
                if part.get("text"):
                    print(part["text"], end="")
    print()

session = await agent.async_create_session(user_id=USER_ID)
session_id = session["id"]
print("Session:", session_id)

await ask(session_id, "I'm shopping for winter running gear. What's your return window?")
await ask(session_id, "And does that same window apply to sale items?")

## 3. Memory Bank spans sessions

The concierge persists each finished turn to **Memory Bank** via its
`after_agent_callback` (see `agent_concierge/utils/memory.py`). Start a brand-new
session for the same user and the agent can recall facts from the earlier one —
`PreloadMemoryTool` loads them into the prompt automatically.

> Memory generation is asynchronous; give it a few moments after the turns above.

In [ ]:
new_session = await agent.async_create_session(user_id=USER_ID)
await ask(new_session["id"], "Based on what you know about me, recommend a category to browse.")

In [ ]:
# Inspect what Memory Bank has distilled for this user.
memories = await agent.async_search_memory(user_id=USER_ID, query="shopping preferences")
memories

## 4. Clean up

Delete the demo sessions. The deployment itself stays up — tear it down with
`make deploy-delete` when you're finished.

In [ ]:
sessions = await agent.async_list_sessions(user_id=USER_ID)
for s in sessions.sessions:
    await agent.async_delete_session(user_id=USER_ID, session_id=s.id)
print("Deleted demo sessions.")

---

### What just happened (Scale + Govern)

- **Sessions** kept multi-turn state server-side (step 2).
- **Memory Bank** carried user facts across *separate* sessions (step 3).
- Behind the scenes, deploying also registered both agents in the
  **[Agent Registry](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/agent-registry)**
  and gave each an **[Agent Identity](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/agent-identity-overview)** — the Govern pillar, on by default.

Next: **Phase 3 (Optimize)** — simulate usage, evaluate quality, and observe traces.